# Simple neural network with Keras

In this tutorial, we will see how to use Keras to build simple neural networks.

It is not necessary to use a graphics card provided by Colab for this tutorial, everything is fast on CPU. You can therefore change the environment if the one you have been assigned has a graphics card (`Run > Change runtime type`).

In [ ]:
!pip install keras-tuner
import datetime
import typing

import keras_tuner
import numpy
import tensorflow
import tensorflow.keras as keras

## Data

We load data directly from Keras.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

## Have a look at the data

In the next cell, study the data structure (shape, size, …).

In [ ]:
# your code here

### Solution

In [ ]:
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"y_train shape : {y_train.shape}")
print(f"y_test shape  : {y_test.shape}")

In [ ]:
print(f"X_train type : {X_train.dtype}")
print(f"y_train type : {y_train[0].dtype}")
print(f"y_train example : {y_train[0]}")

In [ ]:
import matplotlib.pyplot as plt


n = 10
f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
for i in range(n):
    ax[i].imshow(X_train[i], cmap="gray_r")
    ax[i].set_title(y_train[i])
    ax[i].axis("off")
plt.show()

## Transform data

We need to perform some transformations on this data:

1. We could keep the original forms of the numpy arrays `(_, 28, 28)` but it will be easier to work on tensors of form `(_, 28²)` where `_` is the original number of examples.
2. The current type of array is `uint8` as we have just seen. Keras uses `float32` by default. So we need to convert our arrays.
3. We will see two ways to define the loss function. The first one requires the classes to be [one-hot encoded](https://fr.wikipedia.org/wiki/Encodage_one-hot).

Some functions that may be useful:

- [`numpy.ndarray.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html)
- [`numpy.ndarray.astype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html)
- [`tensorflow.keras.utils.to_categorical`](https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical)

*Perform the first two transformations on `X_train` and `X_test` and the last one on `y_train` and `y_test`.*

In [ ]:
# your code here

### Solution

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# We want to put X "flat" so that the input to our network is a vector
# of size input_dim
X_train = X_train.reshape(-1, input_dim)
X_test = X_test.reshape(-1, input_dim)

# X is a numpy.ndarray of uint8. Keras layers expect by default
# float32 entries
X_train = X_train.astype(numpy.float32)
X_test = X_test.astype(numpy.float32)

# Convert targets to sparse vector
Y_train = keras.utils.to_categorical(y_train, nb_classes)
Y_test = keras.utils.to_categorical(y_test, nb_classes)

## Data Normalization

Normalize data with [`numpy.mean`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) and [`numpy.std`](https://numpy.org/doc/stable/reference/generated/numpy.std.html).


In [ ]:
# your code here

### Solution

In [ ]:
# Normalization
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)

# There are constant columns (pixels always at 0). To avoid
# divisions by zero, we can divide by 1 in this case
X_std[X_std == 0] = 1


def normalize(array: numpy.ndarray) -> numpy.ndarray:
  return (array - X_mean) / X_std


X_train = normalize(X_train)
X_test = normalize(X_test)

## Model creation

Create a model without hidden layers that takes an image as input and tries to predict the corresponding class.

Display a summary of this model and explain the numbers you see.

In [ ]:
# model = keras.models.Sequential()
# model.add(???)
# ???
# model.summary()

### Solution

In [ ]:
model = keras.models.Sequential()
model.add(keras.layers.Input((input_dim,))
model.add(keras.layers.Dense(nb_classes, activation="softmax"))
model.summary()

## Training

Perform a training of this model on the data.

We will use :

- 128 as the batch size
- 10 iterations
- 20% of the train base as validation base

In [ ]:
# your code here

### Solution

In [ ]:
# compile is used to "attach" an optimizer, a loss function and
# metrics to a model
model.compile(optimizer="sgd",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# fit uses the "compiled" model in a complete training loop
model.fit(X_train,
          Y_train,
          batch_size=128,
          epochs=10,
          verbose=1,
          validation_split=0.2)


def evaluate(model: keras.models.Model, one_hot: bool) -> None:
  score = model.evaluate(X_test, Y_test if one_hot else y_test, verbose=0)
  print(f"test loss : {score[0]}")
  print(f"test accuracy : {score[1]}")


evaluate(model, True)

Alternative solution that uses a loss that allows `y_train` to be used directly rather than `Y_train` :

In [ ]:
model2 = keras.models.Sequential()
model.add(keras.layers.Input((input_dim,))
model.add(keras.layers.Dense(nb_classes, activation="softmax"))
model2.summary()

model2.compile(optimizer="sgd",
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

model2.fit(X_train,
           y_train,
           batch_size=128,
           epochs=10,
           verbose=1,
           validation_split=0.2)

evaluate(model2, one_hot=False)

Solution with 4 hidden layers of size 20, L2 regularization of the parameters and orthogonal initialization of the weight matrices

In [ ]:
def build_deep_model() -> keras.models.Model:
  hidden_params = dict(activation='relu',
                      kernel_regularizer='l2',
                      bias_regularizer="l2",
                      kernel_initializer='orthogonal')

  model = keras.models.Sequential(name="deep_model")
  model.add(keras.layers.Input((input_dim,))
  model.add(keras.layers.Dense(20, **hidden_params))
  model.add(keras.layers.Dense(20, **hidden_params))
  model.add(keras.layers.Dense(20, **hidden_params))
  model.add(keras.layers.Dense(20, **hidden_params))
  model.add(keras.layers.Dense(nb_classes,
                               activation="softmax",
                               kernel_regularizer="l2",
                               bias_regularizer="l2",
                               kernel_initializer="orthogonal"))
  model.summary()
  return model


deep_model = build_deep_model()

deep_model.compile(optimizer="adam",
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])

deep_model.fit(X_train,
               y_train,
               batch_size=128,
               epochs=10,
               verbose=1,
               validation_split=0.2)
evaluate(deep_model, one_hot=False)

## Hyper-parameters search

To find the optimal hyper-parameters, it is possible to use the [`keras-tuner`](https://keras-team.github.io/keras-tuner/) library. To do this, you need to define a function that creates a model by sampling the parameters. Refer to the example on the Keras Tuner home page to define a model that Keras Tuner can use and then use [the Bayesian Process-based Tuner](https://keras-team.github.io/keras-tuner/documentation/tuners/#bayesianoptimization-class) to find the optimal hyper-parameters for your model.

In [ ]:
# def build_model(hp: keras_tuner.HyperParameters):
#   ???
# tuner = keras_tuner.tuners.bayesian.BayesianOptimization(???)
# tuner.search_space_summary()
# tuner.search(???)

### Solution

In [ ]:
def build_model(hp: keras_tuner.HyperParameters):
  model = keras.models.Sequential()
  model.add(keras.layers.Input((input_dim,))
  model.add(keras.layers.Dense(units=hp.Int('units',
                                            min_value=32,
                                            max_value=512,
                                            step=32),
                               activation='relu'))
  model.add(keras.layers.Dense(nb_classes, activation='softmax'))
  model.compile(
      optimizer=keras.optimizers.Adam(hp.Choice('learning_rate',
                                      values=[1e-2, 1e-3, 1e-4])),
      loss='sparse_categorical_crossentropy',
      metrics=['accuracy'])
  return model


now = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

tuner = keras_tuner.BayesianOptimization(
  build_model,
  objective="val_accuracy",
  max_trials=5,
  executions_per_trial=3,
  directory=f"logs/hp-{now}",
  project_name="mnist")

tuner.search_space_summary()

In [ ]:
tuner.search(X_train, y_train,
             epochs=10,
             batch_size=1500,
             validation_split=0.2)

In [ ]:
tuner.results_summary()
best_models = tuner.get_best_models()
print(best_models[0].summary())

## Using TensorBoard to visualise metrics

Passing a callback [`tensorflow.keras.callbacks.TensorBoard`](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/TensorBoard) to `fit` in its `callbacks` argument enables TensorBoard. You can then view the training directly in Colab using the `tensorboard` extension.

In [ ]:
now = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

deep_model = build_deep_model()

deep_model.compile(optimizer="adam",
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])

tb_callback = keras.callbacks.TensorBoard(log_dir=f"logs/adam-{now}")

deep_model.fit(X_train, y_train, batch_size=3000, epochs=10,
               callbacks=[tb_callback], validation_split=0.2)


# Same model but with SGD as an optimizer
deep_model = build_deep_model()

deep_model.compile(optimizer="sgd",
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])

tb_callback = keras.callbacks.TensorBoard(log_dir=f"logs/sgd-{now}")

deep_model.fit(X_train, y_train, batch_size=3000, epochs=10,
               callbacks=[tb_callback], validation_split=0.2)

%reload_ext tensorboard
%tensorboard --logdir logs